# Notebook 04 – DoubleML for US‑China (Robust Causal Estimation)
**Thesis: Geopolitical Turning Points and Macroeconomic Volatility – Extension of Saadaoui (2026)**  

This notebook uses **Double Machine Learning** to estimate the causal effect of US‑China geopolitical shocks (`lpri` instrumented by `d2pri`) on oil prices (`lwti`) while controlling for a high‑dimensional set of macro, financial, and NLP features.

We implement the **Partially Linear IV (PLIV)** model:
```
lwti_{t+h} = β * lpri_t + g(controls) + ε
```
with `d2pri` as the instrument. The nuisance function `g()` is estimated using regularised ML (Ridge or XGBoost).  
We use 5‑fold cross‑fitting with 3 repetitions to reduce overfitting bias.

**Outputs:**
- `irf_dml_ridge.csv` – impulse response (Ridge learner)
- `irf_dml_xgb.csv` – impulse response (XGBoost learner)
- `diagnostics/` – first‑stage F, stability metrics, placebo test results

---

## 1. Setup and Imports

In [3]:
import numpy as np
import pandas as pd
import json
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

import doubleml as dml
from sklearn.linear_model import RidgeCV, LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

np.random.seed(42)

# ── Robust path resolution ──────────────────────────────────────────────
ROOT = Path.cwd().resolve()
# If running inside 'notebooks' folder, go up one level
if ROOT.name == 'notebooks':
    PROJECT_ROOT = ROOT.parent
else:
    PROJECT_ROOT = ROOT

DATA_NLP = PROJECT_ROOT / 'data' / '03_nlp'
FEAT_MATRIX = DATA_NLP / 'feature_matrix_nlp_A.csv'
VAR_ROLES = DATA_NLP / 'var_roles_nlp_A.json'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Feature matrix exists: {FEAT_MATRIX.exists()}")
print(f"Variable roles exists: {VAR_ROLES.exists()}")

Project root: C:\Users\HP\Desktop\replication+contribution
Feature matrix exists: True
Variable roles exists: True


In [4]:
# Load data
df = pd.read_csv(FEAT_MATRIX, index_col=0, parse_dates=True)
df.index = pd.to_datetime(df.index).to_period('M').to_timestamp()
print(f"Shape: {df.shape}")
print(f"Date range: {df.index.min().date()} to {df.index.max().date()}")

with open(VAR_ROLES, 'r') as f:
    roles = json.load(f)

# Baseline controls (from Saadaoui)
base_controls = roles['controls_baseline']  # llwip, dllgop, l2lwip, dl2lgop

# NLP primary controls (from GDELT + GPR + WUI, without EA‑GPR)
nlp_primary = roles['controls_nlp_primary']   # list of NLP features

# All controls for the primary spec = macro dense + NLP primary
macro_dense = roles['controls_macro_dense']   # from NB02
all_controls = base_controls + macro_dense + nlp_primary

# Ensure all controls exist in df
all_controls = [c for c in all_controls if c in df.columns]
print(f"Number of controls: {len(all_controls)}")

Shape: (386, 88)
Date range: 1990-01-01 to 2022-02-01
Number of controls: 28


In [5]:
# Definition of key horizons (to save computation time)
hmax = 48
key_horizons = [0, 6, 12, 24, 36, 48]   # we will compute all but optionally restrict
compute_all_horizons = True   # set False to only compute key horizons
horizons = range(hmax+1) if compute_all_horizons else key_horizons
print(f"Horizons to compute: {list(horizons)}")

Horizons to compute: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48]


## 2. DoubleML PLIV Estimator Function

We define a function that, for a given horizon `h`, prepares the data and runs `DoubleMLPLIV` with two learners.

In [6]:
from doubleml import DoubleMLData, DoubleMLPLIV
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

def get_ridge_learner():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=1.0))
    ])

def get_xgb_learner():
    return XGBRegressor(
        n_estimators=150, max_depth=3, learning_rate=0.05,
        subsample=0.7, colsample_bytree=0.7,
        verbosity=0, random_state=42, n_jobs=-1
    )

def run_dml_pliv(df, y_col, d_col, z_col, controls, horizon,
                 ml_l, ml_m, ml_r, n_folds=5, n_rep=3):
    """
    Run DoubleML PLIV for a single horizon.
    Returns: (coef, se, first_stage_f, ci_low, ci_high)
    """
    # Prepare data
    work = df.copy()
    work['y_fwd'] = work[y_col].shift(-horizon)
    # Lags of y (outcome) as controls
    for lag in range(1, 4):
        work[f'L{y_col}_{lag}'] = work[y_col].shift(lag)
    # Lags of d (endogenous) as controls
    for lag in range(1, 3):
        work[f'L{d_col}_{lag}'] = work[d_col].shift(lag)
    
    # Combine all exogenous variables
    lag_y = [f'L{y_col}_{l}' for l in range(1,4)]
    lag_d = [f'L{d_col}_{l}' for l in range(1,3)]
    X_cols = lag_y + lag_d + controls
    X_cols = [c for c in X_cols if c in work.columns]
    
    # Drop missing values
    reg_df = work[['y_fwd', d_col, z_col] + X_cols].dropna()
    if len(reg_df) < 80:
        return np.nan, np.nan, np.nan, np.nan, np.nan
    
    # DoubleML data object
    dml_data = DoubleMLData(
        reg_df,
        y_col='y_fwd',
        d_cols=d_col,
        z_cols=z_col,
        x_cols=X_cols
    )
    
    # PLIV model
    dml_pliv = DoubleMLPLIV(
        dml_data,
        ml_l=ml_l,
        ml_m=ml_m,
        ml_r=ml_r,
        n_folds=n_folds,
        n_rep=n_rep
    )
    dml_pliv.fit()
    
    coef = dml_pliv.coef[0]
    se = dml_pliv.se[0]
    # First-stage F statistic (approximated from the first-stage coefficient on z in the linear case? We'll compute manually)
    # For simplicity, we can compute the first-stage F using OLS on the same data (strong instrument)
    # But DoubleML does not directly provide it. We'll compute separately.
    # Compute first-stage F (robust) using OLS for diagnostic
    import statsmodels.api as sm
    X_fs = sm.add_constant(reg_df[[z_col] + X_cols])
    fs_fit = sm.OLS(reg_df[d_col], X_fs).fit(cov_type='HC1')
    f_stat = fs_fit.f_test(f'{z_col}=0').fvalue
    
    ci_low = coef - 1.645 * se
    ci_high = coef + 1.645 * se
    return coef, se, f_stat, ci_low, ci_high

## 3. Run DoubleML for Ridge and XGBoost Learners

We will loop over horizons and collect results.

In [7]:
results_ridge = []
results_xgb = []

for h in horizons:
    print(f"Horizon {h:2d}...", end=' ')
    
    # Ridge
    coef_r, se_r, f_r, lo_r, hi_r = run_dml_pliv(
        df, 'lwti', 'lpri', 'd2pri', all_controls, h,
        ml_l=get_ridge_learner(), ml_m=get_ridge_learner(), ml_r=get_ridge_learner()
    )
    results_ridge.append({
        'h': h, 'coef': coef_r, 'se': se_r, 'F': f_r,
        'lo90': lo_r, 'hi90': hi_r
    })
    
    # XGBoost
    coef_x, se_x, f_x, lo_x, hi_x = run_dml_pliv(
        df, 'lwti', 'lpri', 'd2pri', all_controls, h,
        ml_l=get_xgb_learner(), ml_m=get_xgb_learner(), ml_r=get_xgb_learner()
    )
    results_xgb.append({
        'h': h, 'coef': coef_x, 'se': se_x, 'F': f_x,
        'lo90': lo_x, 'hi90': hi_x
    })
    
    print(f"Ridge: {coef_r:.4f} (se={se_r:.4f})  XGB: {coef_x:.4f} (se={se_x:.4f})")

# Convert to DataFrames
irf_ridge = pd.DataFrame(results_ridge)
irf_xgb = pd.DataFrame(results_xgb)

# Save
irf_ridge.to_csv(RESULTS_DIR / 'irf_dml_ridge.csv', index=False)
irf_xgb.to_csv(RESULTS_DIR / 'irf_dml_xgb.csv', index=False)
print("\nSaved IRF tables.")

Horizon  0... Ridge: -0.0432 (se=0.0395)  XGB: 0.0231 (se=0.0478)
Horizon  1... Ridge: -0.0669 (se=0.0543)  XGB: -0.0070 (se=0.0615)
Horizon  2... Ridge: -0.1092 (se=0.0651)  XGB: -0.0746 (se=0.0650)
Horizon  3... Ridge: -0.1087 (se=0.0649)  XGB: -0.0817 (se=0.0692)
Horizon  4... Ridge: -0.1347 (se=0.0609)  XGB: -0.0692 (se=0.0842)
Horizon  5... Ridge: -0.1164 (se=0.0721)  XGB: -0.0078 (se=0.0669)
Horizon  6... Ridge: -0.1232 (se=0.0767)  XGB: -0.1065 (se=0.0662)
Horizon  7... Ridge: -0.1013 (se=0.0904)  XGB: -0.0995 (se=0.0905)
Horizon  8... Ridge: 0.0158 (se=0.1475)  XGB: -0.0051 (se=0.0807)
Horizon  9... Ridge: -0.0924 (se=0.1140)  XGB: 0.0118 (se=0.0716)
Horizon 10... Ridge: -0.0964 (se=0.0938)  XGB: -0.0327 (se=0.0633)
Horizon 11... Ridge: 0.0829 (se=0.1412)  XGB: 0.0789 (se=0.0682)
Horizon 12... Ridge: 0.0338 (se=0.1248)  XGB: 0.0593 (se=0.0829)
Horizon 13... Ridge: -0.1203 (se=0.1417)  XGB: 0.0352 (se=0.0852)
Horizon 14... Ridge: 0.0163 (se=0.1189)  XGB: 0.0244 (se=0.0757)
Horiz

In [8]:
# Quick plot comparison of Ridge vs XGBoost
plt.figure(figsize=(10,6))
plt.plot(irf_ridge['h'], irf_ridge['coef'], label='Ridge', color='blue')
plt.fill_between(irf_ridge['h'], irf_ridge['lo90'], irf_ridge['hi90'], color='blue', alpha=0.2)
plt.plot(irf_xgb['h'], irf_xgb['coef'], label='XGBoost', color='red')
plt.fill_between(irf_xgb['h'], irf_xgb['lo90'], irf_xgb['hi90'], color='red', alpha=0.2)
plt.axhline(0, color='black', linestyle='--')
plt.xlabel('Horizon (months)')
plt.ylabel('Coefficient on lpri')
plt.title('DoubleML PLIV: Ridge vs XGBoost (90% CI)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'dml_ridge_vs_xgb.png', dpi=200)
plt.close()
print("Saved comparison plot.")

Saved comparison plot.


## 4. Diagnostics and Robustness Checks

### 4.1 Compare with Baseline IV‑LP (from NB01)

In [9]:
# Load baseline IRF from NB01 (if available)
baseline_path = RESULTS_DIR / 'irf_figure4_us_china.csv'
if baseline_path.exists():
    irf_baseline = pd.read_csv(baseline_path)
    plt.figure(figsize=(10,6))
    plt.plot(irf_baseline['h'], irf_baseline['coef'], label='IV‑LP (Saadaoui replication)', color='black', linestyle='--')
    plt.plot(irf_ridge['h'], irf_ridge['coef'], label='DoubleML (Ridge)', color='blue')
    plt.plot(irf_xgb['h'], irf_xgb['coef'], label='DoubleML (XGBoost)', color='red')
    plt.axhline(0, color='black', linestyle='-', linewidth=0.8)
    plt.xlabel('Horizon (months)')
    plt.ylabel('Coefficient')
    plt.title('DoubleML vs Baseline IV‑LP')
    plt.legend()
    plt.grid(alpha=0.3)
    plt.savefig(RESULTS_DIR / 'dml_vs_baseline.png', dpi=200)
    plt.close()
    print("Comparison plot saved.")
else:
    print("Baseline IRF not found – skipping comparison.")

Comparison plot saved.


### 4.2 Placebo Test (Permuted Instrument)

In [10]:
def placebo_test(horizon, n_perm=100):
    # Original coefficient
    coef_orig, se_orig, f_orig, _, _ = run_dml_pliv(
        df, 'lwti', 'lpri', 'd2pri', all_controls, horizon,
        ml_l=get_ridge_learner(), ml_m=get_ridge_learner(), ml_r=get_ridge_learner()
    )
    perm_coefs = []
    for i in range(n_perm):
        df_perm = df.copy()
        df_perm['d2pri_perm'] = np.random.permutation(df_perm['d2pri'].values)
        coef_p, _, _, _, _ = run_dml_pliv(
            df_perm, 'lwti', 'lpri', 'd2pri_perm', all_controls, horizon,
            ml_l=get_ridge_learner(), ml_m=get_ridge_learner(), ml_r=get_ridge_learner()
        )
        if not np.isnan(coef_p):
            perm_coefs.append(coef_p)
    p_val = np.mean(np.abs(perm_coefs) >= np.abs(coef_orig))
    return coef_orig, p_val, perm_coefs

# Test at h=6 (where the effect is strongest in baseline)
h_test = 6
coef_obs, p_val, perm_dist = placebo_test(h_test, n_perm=100)
print(f"Placebo test at h={h_test}: observed coef = {coef_obs:.4f}, p-value = {p_val:.3f}")

plt.hist(perm_dist, bins=20, alpha=0.7, label='Permuted coefficients')
plt.axvline(coef_obs, color='red', linestyle='--', label='Observed')
plt.xlabel('Coefficient')
plt.title(f'Placebo distribution (h={h_test})')
plt.legend()
plt.savefig(RESULTS_DIR / f'dml_placebo_h{h_test}.png')
plt.close()

Placebo test at h=6: observed coef = -0.1129, p-value = 0.970


### 4.3 Overfitting Detection – Cross‑validation performance (optional, can be heavy)
We skip detailed implementation but note in the text.

## 5. Interpretation and Summary

In [11]:
print("\n" + "="*70)
print("DOUBLEML RESULTS SUMMARY – US‑CHINA")
print("="*70)
print(f"Ridge: Significant horizons (90% CI): {(irf_ridge['lo90']>0).sum() + (irf_ridge['hi90']<0).sum()} / {len(irf_ridge)}")
print(f"XGBoost: Significant horizons: {(irf_xgb['lo90']>0).sum() + (irf_xgb['hi90']<0).sum()} / {len(irf_xgb)}")
print("\nKey horizon coefficients:")
print("   h   Ridge      XGBoost")
key_h = [0,6,12,24,36,48]
for h in key_h:
    r = irf_ridge[irf_ridge['h']==h].iloc[0]
    x = irf_xgb[irf_xgb['h']==h].iloc[0]
    print(f"  {h:2d}   {r['coef']:.4f} ({r['se']:.4f})   {x['coef']:.4f} ({x['se']:.4f})")


DOUBLEML RESULTS SUMMARY – US‑CHINA
Ridge: Significant horizons (90% CI): 9 / 49
XGBoost: Significant horizons: 1 / 49

Key horizon coefficients:
   h   Ridge      XGBoost
   0   -0.0432 (0.0395)   0.0231 (0.0478)
   6   -0.1232 (0.0767)   -0.1065 (0.0662)
  12   0.0338 (0.1248)   0.0593 (0.0829)
  24   0.1105 (0.0836)   -0.0222 (0.0632)
  36   0.1814 (0.1009)   -0.0301 (0.0903)
  48   -0.2360 (0.2023)   -0.0068 (0.0757)


## 6. Save Full Results and End

In [12]:
print("\nAll results saved in:", RESULTS_DIR)
print("Notebook completed successfully.")


All results saved in: C:\Users\HP\Desktop\replication+contribution\results
Notebook completed successfully.
